# Heatwave Prediction — ERA5 4 Years + Adapted ResNet-34 (v3)
**2022–2025 (Mar–Jun) | ~1211 sequences | Chronological split | Leakage-free**

| Step | Description | Fix applied |
|------|-------------|-------------|
| 1 | Load & merge 4 years of ERA5 NetCDF | — |
| 2 | Daily features: tmax, tmean, d2m_mean, sp_mean | — |
| 3 | Heatwave label: tmax > 35 °C (3×3 neighbourhood avg) | Neighbourhood avg instead of single pixel |
| 4 | 7-day sliding window → (N, 28, 7, 7) | — |
| 5 | Chronological split, per-channel normalise (train stats only) | Fixed data leakage + temporal leakage |
| 6 | pos_weight loss for imbalance | — |
| 7 | Adapted ResNet-34 (3×3 first conv, no MaxPool, binary head) | Modified for 7×7 spatial input |
| 8 | Train with ReduceLROnPlateau + val monitoring | Smarter scheduler |
| 9 | Threshold tuning on val set, final eval on test set | Fixed threshold leakage |


## Step 0 — Imports

In [ ]:
import numpy as np
import xarray as xr
import torch
import torch.nn as nn
import torch.utils.data as data_utils
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    brier_score_loss, roc_curve, auc
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
print("Imports OK")

## Step 1 — Load & Merge 4 Years of ERA5 Data

In [ ]:
# Adjust paths for your environment
PATH_2022 = "data_stream-oper_stepType-instant-2.nc"
PATH_2023 = "data_stream-oper_stepType-instant.nc"
PATH_2024 = "data_stream-oper_stepType-instant-3.nc"
PATH_2025 = "data_stream-oper_stepType-instant-4.nc"

ds = xr.concat(
    [xr.open_dataset(p) for p in [PATH_2022, PATH_2023, PATH_2024, PATH_2025]],
    dim="valid_time"
).sortby("valid_time")  # ensures strict chronological order

print("Merged time range:", ds.valid_time.min().values, "to", ds.valid_time.max().values)
print("Variables        :", list(ds.data_vars))
print("Grid shape       :", ds.t2m.shape, "-> (time, lat=7, lon=7)")

## Step 2 — Daily Feature Engineering

In [ ]:
t2m_c    = ds["t2m"] - 273.15   # Kelvin -> Celsius
d2m_c    = ds["d2m"] - 273.15

tmax     = t2m_c.resample(valid_time="1D").max()
tmean    = t2m_c.resample(valid_time="1D").mean()
d2m_mean = d2m_c.resample(valid_time="1D").mean()
sp_mean  = ds["sp"].resample(valid_time="1D").mean()

print("Daily shape:", tmax.shape)
print(f"Temp range : {float(tmean.min()):.1f} C  to  {float(tmax.max()):.1f} C")

## Step 3 — Heatwave Label

**Fix:** Instead of a single centre pixel (brittle to spatial noise), we average
`tmax` over a 3×3 neighbourhood centred at (3,3) before applying the threshold.
This makes the label more robust to small spatial shifts in the heat centre.

**Criterion:** neighbourhood-mean tmax > 35 °C — IMD plains threshold.


In [ ]:
CX, CY    = 3, 3          # centre indices
THRESHOLD = 35.0

# 3×3 neighbourhood: rows 2-4, cols 2-4
tmax_neighbourhood = tmax[:, CX-1:CX+2, CY-1:CY+2].mean(dim=["latitude", "longitude"])
y = (tmax_neighbourhood > THRESHOLD).values.astype(int)

n_hw  = y.sum()
n_tot = len(y)
print(f"Total days   : {n_tot}")
print(f"Heatwave (1) : {n_hw}  ({100*n_hw/n_tot:.1f}%)")
print(f"Normal   (0) : {n_tot-n_hw}  ({100*(n_tot-n_hw)/n_tot:.1f}%)")
print(f"Imbalance    ~ {(n_tot-n_hw)/max(n_hw,1):.1f} : 1")

## Step 4 — 7-Day Sliding Window Sequences

In [ ]:
WINDOW = 7

X_seq, y_seq = [], []

for i in range(WINDOW, len(tmax)):
    sample = []
    for j in range(i - WINDOW, i):
        sample.append(tmax[j].values)
        sample.append(tmean[j].values)
        sample.append(d2m_mean[j].values)
        sample.append(sp_mean[j].values)
    X_seq.append(np.stack(sample))   # shape: (28, 7, 7)
    y_seq.append(y[i])

X_seq = np.array(X_seq)   # (N, 28, 7, 7)
y_seq = np.array(y_seq)   # (N,)

print(f"X_seq: {X_seq.shape}  -> (samples, 28 channels, 7x7 grid)")
print(f"Labels — 0: {(y_seq==0).sum()}, 1: {(y_seq==1).sum()}")

## Step 5 — Chronological Split + Per-Channel Normalisation

### Fixes applied here

**Fix 1 — Temporal leakage:** We split chronologically (first 70% train,
next 10% val, last 20% test). Random shuffling mixes future data into training,
which gives falsely optimistic metrics on time-series data.

**Fix 2 — Data leakage:** Normalisation statistics (mean, std) are computed
from the *training set only*, then applied to val and test. Computing stats
from the full dataset before splitting leaks test information into training.

**Fix 3 — Per-channel normalisation:** `sp` is in Pascals (~100 000 Pa) while
temperatures are in tens of °C. Global normalisation lets sp dominate. We
normalise each of the 28 channels independently.


In [ ]:
# --- replace NaN / inf before anything else ---
X_seq = np.nan_to_num(X_seq, nan=0.0, posinf=0.0, neginf=0.0)

N = len(X_seq)
train_end = int(N * 0.70)
val_end   = int(N * 0.80)

X_train = X_seq[:train_end]
X_val   = X_seq[train_end:val_end]
X_test  = X_seq[val_end:]

y_train = y_seq[:train_end]
y_val   = y_seq[train_end:val_end]
y_test  = y_seq[val_end:]

# Per-channel Z-score: axis=(0,2,3) collapses samples + spatial dims
ch_mean = X_train.mean(axis=(0, 2, 3), keepdims=True)   # (1, 28, 1, 1)
ch_std  = X_train.std(axis=(0, 2, 3),  keepdims=True) + 1e-8

X_train = (X_train - ch_mean) / ch_std
X_val   = (X_val   - ch_mean) / ch_std   # train stats applied to val
X_test  = (X_test  - ch_mean) / ch_std   # train stats applied to test

print(f"Train : {X_train.shape}  | 0: {(y_train==0).sum()}  1: {(y_train==1).sum()}")
print(f"Val   : {X_val.shape}    | 0: {(y_val==0).sum()}    1: {(y_val==1).sum()}")
print(f"Test  : {X_test.shape}   | 0: {(y_test==0).sum()}   1: {(y_test==1).sum()}")
print(f"Train mean (should be ~0): {X_train.mean():.5f}")
print(f"Train std  (should be ~1): {X_train.std():.5f}")

## Step 6 — Handle Imbalance with `pos_weight`

`pos_weight = n_negative / n_positive` multiplies the loss for heatwave examples
so the model is penalised more for missing them — no data duplication needed.


In [ ]:
n0 = (y_train == 0).sum()
n1 = (y_train == 1).sum()
pos_weight = torch.tensor([n0 / max(n1, 1)], dtype=torch.float32)

print(f"pos_weight: {pos_weight.item():.2f}  (train 0:{n0}, 1:{n1})")

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

## Step 7 — Adapted ResNet-34 for 7×7 Input

Standard ResNet-34 cannot work on a 7×7 grid as-is:
- Its first conv uses a **7×7 kernel, stride 2** → spatial size drops to 3×3 immediately
- Then **MaxPool(3×3, stride 2)** → drops to 1×1 — all spatial info destroyed

**Adaptations made:**
1. First conv: 7×7 stride 2 → **3×3 stride 1**, preserves spatial resolution
2. MaxPool after first conv: **removed entirely**
3. Input channels: 3 → **28** (7 days × 4 variables)
4. Final FC layer: 1000-class → **binary (1 output)**
5. All residual blocks unchanged — full depth of ResNet-34 is retained


In [ ]:
import torchvision.models as models

def build_adapted_resnet34(in_channels=28):
    """
    ResNet-34 adapted for (N, 28, 7, 7) input:
      - First conv: 7x7 stride-2 -> 3x3 stride-1  (preserves 7x7 spatial map)
      - MaxPool removed                             (would collapse 7x7 to 1x1)
      - in_channels: 3 -> 28                       (7 days x 4 variables)
      - FC head:    1000 -> 1                      (binary classification)
    """
    model = models.resnet34(weights=None)  # random init, no ImageNet weights

    # Fix 1: replace first conv — keep same out_channels(64) and padding
    model.conv1 = nn.Conv2d(
        in_channels, 64,
        kernel_size=3, stride=1, padding=1, bias=False
    )

    # Fix 2: remove MaxPool (nn.Identity passes input through unchanged)
    model.maxpool = nn.Identity()

    # Fix 3: replace classification head for binary output
    model.fc = nn.Linear(model.fc.in_features, 1)

    return model


model = build_adapted_resnet34(in_channels=28)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

# Quick sanity check — forward pass with a dummy batch
with torch.no_grad():
    dummy = torch.zeros(4, 28, 7, 7)
    out   = model(dummy)
    print(f"Output shape : {out.shape}  (expected: torch.Size([4, 1]))")


## Step 8 — Training with ReduceLROnPlateau + Validation Monitoring

**Fix:** Replaced `StepLR` (halves LR every 10 epochs blindly) with
`ReduceLROnPlateau` which reduces LR only when validation loss stops
improving — far more data-driven for a 30-epoch run.

We track validation loss each epoch and save the best checkpoint.


In [ ]:
EPOCHS     = 40
BATCH_SIZE = 32

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# ReduceLROnPlateau: halve LR if val loss doesn't improve for 4 epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=4
)

dataset = data_utils.TensorDataset(X_train_t, y_train_t)
loader  = data_utils.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

best_val_loss = float('inf')
best_state    = None
train_losses  = []
val_losses    = []

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        out  = model(xb).squeeze()
        loss = criterion(out, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(loader)
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    with torch.no_grad():
        val_out  = model(X_val_t).squeeze()
        val_loss = criterion(val_out, y_val_t).item()
    val_losses.append(val_loss)

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    scheduler.step(val_loss)

    if (epoch + 1) % 5 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:2d}/{EPOCHS}  Train Loss: {avg_train_loss:.4f}  Val Loss: {val_loss:.4f}  LR: {lr:.6f}")

# Restore best weights
model.load_state_dict(best_state)
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
torch.save(model.state_dict(), 'climatenet_best.pth')
print("Model saved to climatenet_best.pth")

## Step 8b — Training & Validation Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train loss', color='steelblue')
plt.plot(val_losses,   label='Val loss',   color='tomato')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.tight_layout()
plt.show()

## Step 9 — Threshold Tuning on Validation Set

**Fix:** Threshold is tuned on the *validation set* (unseen during training),
not on the test set. Tuning on the test set is a form of data leakage that
inflates F1. Final metrics are reported on the *test set only*.


In [ ]:
model.eval()

# --- Tune threshold on VALIDATION set ---
with torch.no_grad():
    val_probs = torch.sigmoid(model(X_val_t).squeeze()).numpy()

y_val_np = y_val_t.numpy()
thresholds = np.arange(0.20, 0.80, 0.02)
best_t, best_f1 = 0.5, 0.0

for t in thresholds:
    f1 = f1_score(y_val_np, (val_probs > t).astype(int), zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print(f"Best threshold (val)  : {best_t:.2f}")
print(f"Best F1 on val set    : {best_f1:.4f}")

# --- Final evaluation on TEST set ---
with torch.no_grad():
    test_probs = torch.sigmoid(model(X_test_t).squeeze()).numpy()

y_true       = y_test_t.numpy()
final_preds  = (test_probs > best_t).astype(int)

print()
print("=" * 40)
print("      FINAL METRICS  (test set)")
print("=" * 40)
print(f"Accuracy    : {accuracy_score(y_true, final_preds):.4f}")
print(f"Precision   : {precision_score(y_true, final_preds, zero_division=0):.4f}")
print(f"Recall      : {recall_score(y_true, final_preds, zero_division=0):.4f}")
print(f"F1-score    : {f1_score(y_true, final_preds, zero_division=0):.4f}")
print(f"AUC-ROC     : {roc_auc_score(y_true, test_probs):.4f}")
print(f"Brier Score : {brier_score_loss(y_true, test_probs):.4f}")
print()
print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_true, final_preds))

## Step 10 — ROC Curve & Threshold Plot

In [ ]:
fpr, tpr, _ = roc_curve(y_true, test_probs)
roc_auc     = auc(fpr, tpr)

# F1 curve over thresholds — computed on TEST set for display only
f1_vals_test = [f1_score(y_true, (test_probs > t).astype(int), zero_division=0) for t in thresholds]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ROC curve
axes[0].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve (test set)")
axes[0].legend(loc="lower right")

# F1 vs threshold — show val-tuned threshold
axes[1].plot(thresholds, f1_vals_test, color="tomato", lw=2, label="F1 on test")
axes[1].axvline(best_t, linestyle="--", color="gray", label=f"Val-tuned t={best_t:.2f}")
axes[1].set_xlabel("Decision Threshold")
axes[1].set_ylabel("F1-score")
axes[1].set_title("F1-score vs Threshold")
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 11 — Per-Year Breakdown (optional diagnostic)

Split the test predictions by year to see if the model generalises to 2025 or
overfits to earlier years. If 2025 performance is much worse, you may need
more data or a longer window.


In [ ]:
# Extract year for each test sample
# The test window starts at index val_end in the original tmax array
tmax_times = tmax.valid_time.values  # daily timestamps after resample
test_times = tmax_times[val_end + WINDOW : val_end + WINDOW + len(y_test)]

import pandas as pd
test_years = pd.DatetimeIndex(test_times).year

print("Per-year test performance:")
print(f"{'Year':<8} {'N':>6} {'HW':>6} {'Acc':>8} {'F1':>8} {'AUC':>8}")
print("-" * 50)

for yr in sorted(set(test_years)):
    mask = test_years == yr
    if mask.sum() == 0 or len(set(y_true[mask])) < 2:
        continue
    acc = accuracy_score(y_true[mask], final_preds[mask])
    f1  = f1_score(y_true[mask], final_preds[mask], zero_division=0)
    try:
        a = roc_auc_score(y_true[mask], test_probs[mask])
    except Exception:
        a = float('nan')
    n_hw_yr = y_true[mask].sum()
    print(f"{yr:<8} {mask.sum():>6} {n_hw_yr:>6} {acc:>8.3f} {f1:>8.3f} {a:>8.3f}")

## Summary of Changes (v1 → v3)

| # | What changed | Why it matters |
|---|--------------|----------------|
| 1 | **Chronological split** (70/10/20) instead of random | Eliminates temporal leakage |
| 2 | **Per-channel Z-score** using train stats only | Eliminates data leakage + fixes sp scale dominance |
| 3 | **Threshold tuned on val set**, not test set | Eliminates evaluation leakage on F1 |
| 4 | **ReduceLROnPlateau** instead of StepLR | LR adapts to actual validation progress |
| 5 | **Best checkpoint saved** (lowest val loss) | Returns the model that generalised best |
| 6 | **3×3 neighbourhood label** instead of single pixel | More robust heatwave criterion |
| 7 | **Adapted ResNet-34** (3×3 first conv, no MaxPool, binary head) | Full ResNet-34 depth on a 7×7 grid without destroying spatial info |
| 8 | **Per-year diagnostic** (Step 11) | Reveals if 2024–2025 generalisation differs from earlier years |
